In [9]:
import itertools
from collections import defaultdict
import math


In [10]:
class HMMPOSTagger:
    def __init__(self):
        self.tags = set()
        self.words = set()
        self.initial_probs = {}
        self.transition_probs = {}
        self.emission_probs = {}

    def train(self, training_data):
        tag_counts = defaultdict(int)
        init_tag_counts = defaultdict(int)
        transition_counts = defaultdict(lambda: defaultdict(int))
        emission_counts = defaultdict(lambda: defaultdict(int))

        for sentence in training_data:
            prev_tag = None
            for i, (word, tag) in enumerate(sentence):
                self.tags.add(tag)
                self.words.add(word.lower())
                tag_counts[tag] += 1
                emission_counts[tag][word.lower()] += 1

                if i == 0:
                    init_tag_counts[tag] += 1
                if prev_tag:
                    transition_counts[prev_tag][tag] += 1
                prev_tag = tag

        num_tags = len(self.tags)
        num_sents = len(training_data)

        self.initial_probs = {
            tag: (init_tag_counts[tag] + 1) / (num_sents + num_tags)
            for tag in self.tags
        }

        self.transition_probs = {
            t1: {
                t2: (transition_counts[t1][t2] + 1) / (tag_counts[t1] + num_tags)
                for t2 in self.tags
            }
            for t1 in self.tags
        }

        vocab_size = len(self.words)
        self.emission_probs = {
            tag: {
                word: (emission_counts[tag][word] + 1) / (tag_counts[tag] + vocab_size)
                for word in self.words
            }
            for tag in self.tags
        }

    def brute_force_decode(self, sentence):
        sentence = [w.lower() for w in sentence]
        all_tag_sequences = itertools.product(self.tags, repeat=len(sentence))

        best_sequence = None
        best_log_prob = -math.inf

        for seq in all_tag_sequences:
            log_prob = math.log(self.initial_probs.get(seq[0], 1e-10)) + \
                       math.log(self.emission_probs[seq[0]].get(sentence[0], 1e-10))

            for i in range(1, len(sentence)):
                prev_tag, curr_tag = seq[i - 1], seq[i]
                trans_prob = self.transition_probs[prev_tag].get(curr_tag, 1e-10)
                emit_prob = self.emission_probs[curr_tag].get(sentence[i], 1e-10)
                log_prob += math.log(trans_prob) + math.log(emit_prob)

            if log_prob > best_log_prob:
                best_log_prob = log_prob
                best_sequence = seq

        return list(best_sequence)

training_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]



test_sentences = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
]



In [11]:
hmm = HMMPOSTagger()
hmm.train(training_data)

for sent in test_sentences:
    print(sent, "→", hmm.brute_force_decode(sent))


['The', 'dog', 'barks'] → ['DET', 'NOUN', 'VERB']
['A', 'cat', 'sleeps'] → ['DET', 'NOUN', 'VERB']
['The', 'big', 'dog', 'runs'] → ['DET', 'ADJ', 'NOUN', 'VERB']
['Dogs', 'bark'] → ['NOUN', 'VERB']
